# Giải thích Code Question 6: Risk-Aware Evaluation Function (Hàm đánh giá nhận thức rủi ro)

Trong câu hỏi này, chúng ta cần tạo ra một **Hàm đánh giá rủi ro (Risk-Aware Evaluation Function)** cho Pac-Man. 

## Tổng quan về Hàm Evaluation Function

**1. Mục đích của hàm này là gì?**
Hàm này giúp Pac-Man không chỉ biết tìm thức ăn hay nhìn xem ma cách bao xa, mà nó còn biết **đánh giá địa hình**. Nếu nó thấy một viên thức ăn nằm trong một cái hẻm cụt và có ma đang lảng vảng bên ngoài, nó sẽ nhận ra rủi ro bị kẹt cực kỳ cao và từ chối đi vào đó. Hàm này giúp Pac-Man cân bằng giữa lòng tham và sự an toàn.

**2. Nó nhận vào tham số gì?**
Hàm `riskAwareEvaluationFunction(currentGameState)` nhận vào 1 tham số chính:
*   `currentGameState` (Trạng thái hiện tại): Bức tranh toàn cảnh của bàn cờ lúc này (tọa độ Pac-Man, tọa độ ma, danh sách thức ăn, sơ đồ mảng tường...).
Hàm này đánh giá trực tiếp sự tốt hay xấu của một trạng thái cụ thể. 

**3. Nó trả về cái gì?**
Hàm này trả về một **con số**. 
*   Số càng **lớn**, nghĩa là trạng thái hiện tại càng an toàn, gần thức ăn và điểm cao.
*   Số càng **nhỏ**, nghĩa là trạng thái đó rất xấu. Pacman bị dồn vào chân tường và ma đang đến gần.

---

Mục tiêu cốt lõi của hàm đánh giá này gồm 2 phần chính: 
1. Sử dụng thuật toán BFS để tính toán chính xác khoảng cách đi bộ thực tế.
2. Dựa vào kết quả BFS để đánh giá rủi ro địa hình và đưa ra quyết định hành động.

Dưới đây là phần giải thích chi tiết cho từng đoạn code:

## 1. Thuật toán BFS

In [ ]:
DOF_RADIUS = 5

queue = util.Queue()
queue.push((pos, 0))
visited = {pos}

closest_food_dist = None
dof = 1
ghost_dists = [-1] * len(ghostStates) 

while not queue.isEmpty():
    cur_pos, depth = queue.pop()

    # 1. Check for food
    if closest_food_dist is None and cur_pos in foodList:
        closest_food_dist = depth

    # 2. Check for ghosts
    for i in range(len(ghostStates)):
        ghost_pos = ghostStates[i].getPosition()
        ghost_pos_int = (int(ghost_pos[0]), int(ghost_pos[1]))
        if cur_pos == ghost_pos_int and ghost_dists[i] == -1:
            ghost_dists[i] = depth

    # 3. Check stopping conditions
    food_done = False
    if closest_food_dist is not None or len(foodList) == 0:
        food_done = True
        
    dof_done = False
    if depth >= DOF_RADIUS:
        dof_done = True
        
    ghosts_done = True
    for dist in ghost_dists:
        if dist == -1:
            ghosts_done = False
            break

    if food_done and dof_done and ghosts_done:
        break

    # 4. Flood fill to 4 directions
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        next_x = cur_pos[0] + dx
        next_y = cur_pos[1] + dy
        next_pos = (next_x, next_y)
        if next_pos not in visited and not walls[next_x][next_y]:
            visited.add(next_pos)
            queue.push((next_pos, depth + 1))
            if depth + 1 <= DOF_RADIUS:
                dof += 1

if closest_food_dist is None:
    closest_food_dist = 0

*   **Ý nghĩa:** Khoảng cách Manhattan rất hay đánh lừa Pac-Man. Thuật toán BFS được dùng để dò đường thực tế. Vòng lặp BFS này được thiết kế để hoàn thành 3 công việc cùng một lúc:
    1. Tìm khoảng cách đi bộ thực tế đến thức ăn gần nhất (`closest_food_dist`).
    2. Đếm số lượng ô trống xung quanh trong bán kính 5 bước (`dof` - Degrees of Freedom) để đánh giá không gian rộng hay hẹp.
    3. Tìm khoảng cách đi bộ thực tế đến từng con ma trên bản đồ và lưu vào mảng `ghost_dists`.
*   **Giải thích code:**
    *   **Nhiệm vụ 1 - Tìm thức ăn:** Trong quá trình duyệt, nếu phát hiện ô hiện tại chứa thức ăn và đây là viên đầu tiên nhìn thấy, hệ thống sẽ ghi lại số bước đi bộ vào biến `closest_food_dist`.
    *   **Nhiệm vụ 2 - Đo lường Độ tự do:** Khi thuật toán loang ra các hướng xung quanh, nếu ô tiếp theo là ô trống và vẫn nằm trong bán kính quy định (`DOF_RADIUS`), biến `dof` sẽ được cộng thêm 1.
    *   **Nhiệm vụ 3 - Tìm ma:** Cứ mỗi bước đi tới 1 ô, nó quét qua danh sách ma. Nếu phát hiện có con ma đang đứng ở ô đó và chưa được ghi nhận, nó ghi khoảng cách vào mảng `ghost_dists`.
    *   **Điều kiện dừng:** Vòng lặp sẽ tự động ngắt khi cả 3 biến cờ (`food_done`, `dof_done`, `ghosts_done`) đều mang giá trị True.

## 2. Khuyến khích Ăn Thức Ăn và Viên Sức Mạnh

In [ ]:
# 1. Food Score
if len(foodList) > 0:
    food_distance_bonus = 10.0 / (closest_food_dist + 1)
    score += food_distance_bonus
    
    food_remaining_penalty = 4 * len(foodList)
    score -= food_remaining_penalty

# 2. Capsule Score
capsule_penalty = 20 * len(capsules)
score -= capsule_penalty

*   **Ý nghĩa:** Điều hướng Pac-Man tích cực tìm kiếm thức ăn và dọn dẹp viên sức mạnh.
*   **Giải thích code:**
    *   **Thưởng lại gần thức ăn (`food_distance_bonus`):** Lấy `10.0` chia cho khoảng cách. Khoảng cách càng nhỏ, điểm thưởng càng lớn. Khoảng cách này được lấy từ BFS.
    *   **Phạt thức ăn thừa (`food_remaining_penalty`):** Càng sót nhiều hạt, bị trừ càng nhiều điểm. Kích thích Pac-Man ăn sạch bản đồ thật nhanh.
    *   **Phạt viên sức mạnh (`capsule_penalty`):** Trừ một lượng điểm lớn (`20`) cho mỗi viên capsule còn sót lại. Điều này khiến Pac-Man ưu tiên ăn capsule trước.

## 3. Đánh giá Mối Đe Dọa Cơ Bản Của Ma

In [ ]:
for i in range(len(ghostStates)):
    ghost = ghostStates[i]
    
    if ghost_dists[i] != -1:
        ghost_dist = ghost_dists[i]
    else:
        ghost_dist = manhattanDistance(pos, ghost.getPosition())

    if ghost.scaredTimer > 0:
        scared_ghost_bonus = 200.0 / (ghost_dist + 1)
        score += scared_ghost_bonus
    else:
        if ghost_dist <= 1:
            score -= 500  
        else:
            active_ghost_penalty = 2.0 / ghost_dist
            score -= active_ghost_penalty

*   **Ý nghĩa:** Khi ma bị ăn capsule và hoảng sợ, Pac-Man trở thành "thợ săn". Khi ma bình thường, Pac-Man là "con mồi" phải lùi ra xa. Việc sử dụng khoảng cách đi bộ thực tế giúp Pac-Man không hoảng loạn vô lý khi con ma ở sát bên nhưng lại bị ngăn cách bởi một bức tường dài.
*   **Giải thích code:**
    *   Đầu tiên, lấy khoảng cách thực tế từ mảng BFS: `ghost_dists[i]`. Nếu BFS không tìm thấy ma, code sẽ tự động dùng đường chim bay `manhattanDistance` làm phương án dự phòng.
    *   **Nếu ma bị sợ (`scaredTimer > 0`):** Cung cấp một lượng điểm thưởng `scared_ghost_bonus` lớn. Ma càng gần, điểm thưởng càng cao, kích thích Pac-Man đuổi theo ăn con ma đó.
    *   **Nếu ma đang bình thường:** Nếu khoảng cách <= 1, trừ thẳng `500` điểm vì cái chết gần như chắc chắn. Nếu xa hơn, áp dụng hình phạt nhỏ `active_ghost_penalty` để luôn giữ khoảng cách an toàn.

## 4. Nhận Thức Rủi Ro Mắc Kẹt và Tìm Đường Thoát

In [ ]:
MAX_DOF = (DOF_RADIUS * 2 + 1) ** 2  
dof_ratio = dof / MAX_DOF
narrowness_factor = 1.0 - dof_ratio 

THREAT_RADIUS = 6
ghost_threat = 0.0
for i in range(len(ghostStates)):
    ghost = ghostStates[i]
    if ghost.scaredTimer == 0:  
        if ghost_dists[i] != -1:
            dist = ghost_dists[i]
        else:
            dist = manhattanDistance(pos, ghost.getPosition())
            
        if dist < THREAT_RADIUS:  
            ghost_threat += (THREAT_RADIUS - dist)  

entrapment_risk = ghost_threat * narrowness_factor * 40.0
score -= entrapment_risk

if ghost_threat > 0:
    escape_bonus = 0.5 * dof
    score += escape_bonus

*   **Ý nghĩa:** Đoạn code này giúp Pac-man có khả năng nhận thức nguy cơ bị dồn vào đường cùng (Entrapment Risk). Nguyên lý là: **Không gian hẹp + Ma tiến lại gần = Rủi ro cao**.
*   **Giải thích code:**
    *   **Hệ số chật hẹp (`narrowness_factor`):** Pac-Man lấy DoF hiện tại chia cho không gian lý tưởng tối đa (`MAX_DOF`) để biết độ thoáng. Lấy 1 trừ đi số đó sẽ ra mức độ "chật hẹp". Nếu đang đứng ở ngõ cụt thì hệ số này tiến gần về 1.0.
    *   **Sức ép từ ma (`ghost_threat`):** Quét qua các con ma. Con ma nào đang hoạt động và nằm trong bán kính 6 bước đi bộ thì sẽ tạo ra sức ép đe dọa. Ma càng đứng sát, sức ép càng lớn.
    *   **Rủi ro kẹt (`entrapment_risk`):** Tích của sức ép ma nhân với độ chật hẹp địa hình. **Nếu đang bị ma ép sát và đứng trong ngõ cụt, mức trừ phạt sẽ rất cao**. Điều này ngăn Pac-Man đi vào hẻm cụt khi ma rình ở cửa.
    *   **Thưởng tẩu thoát (`escape_bonus`):** Khi nhận thấy có sự đe dọa từ ma (`ghost_threat > 0`), Pac-Man được cộng thêm điểm thưởng tỉ lệ thuận với biến `dof`. Động cơ này lập trình cho Pac-Man luôn hướng về phía các ngã ba, ngã tư rộng rãi để dễ dàng xoay sở và tẩu thoát.

**Tóm lại:** Với hàm đánh giá này, Pac-Man có thể quan sát địa hình xung quanh, ước lượng đường đi bộ thực tế tới thức ăn và đánh giá mức độ nguy hiểm của khu vực. Nó sẵn sàng từ bỏ vài hạt thức ăn lẻ nếu nhận ra đó là một cái bẫy hẻm cụt, đồng thời luôn biết chọn chỗ rộng rãi mỗi khi ma đang lao tới!